In [1]:
import sys
from pathlib import Path
from pyprojroot import here

sys.path.append(str(here()))

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils import load_data
from sklearn.model_selection import RepeatedStratifiedKFold
from src.processing import build_pipeline
from sklearn.pipeline import Pipeline
from tqdm.auto import tqdm
from sklearn.base import clone
from sklearn.inspection import permutation_importance
from sklearn.metrics import root_mean_squared_error
import xgboost as xgb
import lightgbm as lgb
from sklearn.linear_model import LinearRegression
from src.utils import log_fe_experiment
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from catboost import CatBoostRegressor
from src.processing import new_ohe_columns
from xgboost import XGBRegressor
from IPython.display import display
import optuna
from sklearn.model_selection import train_test_split
import os

from src.config import cfg
from src.utils import set_seed

In [3]:
import warnings

warnings.filterwarnings("ignore", message="Found unknown categories in columns")

In [4]:
%load_ext autoreload
%autoreload 2

In [5]:
gseed = cfg.general.seed
set_seed(gseed)

In [6]:
train_path = Path(cfg.paths.train)
test_path = Path(cfg.paths.test)

df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)
df_all_data = pd.concat(
    [df_train.drop(columns=["SalePrice"]), df_test], axis=0
).reset_index(drop=True)

In [7]:
X_train, y_train_raw, X_test, test_ids = load_data(cfg, False)

y_train = np.log1p(y_train_raw)

y_binned = pd.qcut(y_train, q=10, labels=False)

rskf = RepeatedStratifiedKFold(n_splits=10, n_repeats=5, random_state=gseed)

In [8]:
final_tree_pipe = build_pipeline("final_tree")

In [ ]:
def prepare_early_stopping_params(model, X_val, y_val, early_stopping_callback=None):
    model_name = type(model).__name__
    fit_params = {}

    if "LGBM" in model_name:
        fit_params["model__eval_X"] = X_val
        fit_params["model__eval_y"] = y_val

        if early_stopping_callback is not None:
            fit_params["model__callbacks"] = [early_stopping_callback]
        else:
            import lightgbm as lgb

            fit_params["model__callbacks"] = [
                lgb.early_stopping(stopping_rounds=100, verbose=False)
            ]

    elif "XGB" in model_name:
        fit_params["model__eval_set"] = [(X_val, y_val)]

    elif "CatBoost" in model_name:
        fit_params["model__eval_set"] = (X_val, y_val)
        fit_params["model__early_stopping_rounds"] = 100
        fit_params["model__verbose"] = False

    elif "HistGradientBoosting" in model_name:
        fit_params["model__X_val"] = X_val
        fit_params["model__y_val"] = y_val

    elif "GradientBoosting" in model_name:
        pass

    elif "MLP" in model_name:
        fit_params["model__eval_X"] = X_val
        fit_params["model__eval_y"] = y_val

    return fit_params


In [10]:
def cv_result(
    model,
    X_train,
    y_train,
    y_strat,
    cv_splitter,
    preprocessor,
    name=None,
    fit_params=None,
    use_early_stopping=False,
    early_stopping_callback=None,
    calculate_importance=False,
    perm_n_repeats=5,
    random_state=gseed,
):
    """
    Universal cv function, but could use LGBM early stopping
    """
    fit_params = fit_params or {}
    model_pipe = Pipeline([("preprocessor", preprocessor), ("model", model)])

    fitted_models = []
    oof_preds_sum = np.zeros(len(X_train), dtype=float)
    oof_counts = np.zeros(len(X_train), dtype=float)

    tr_scores, val_scores = [], []

    fold_importances = []

    splits = list(cv_splitter.split(X_train, y_strat))

    for train_idx, val_idx in tqdm(
        splits, desc=name or "CV folds", unit="fold", leave=False
    ):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        fold_pipe = clone(model_pipe)

        fold_fit_params = fit_params.copy()

        if use_early_stopping:
            fold_preprocessor = clone(preprocessor)
            X_tr_trans = fold_preprocessor.fit_transform(X_tr, y_tr)
            X_val_trans = fold_preprocessor.transform(X_val)

            framework_params = prepare_early_stopping_params(
                model=model,
                X_val=X_val_trans,
                y_val=y_val,
                early_stopping_callback=early_stopping_callback,
            )
            fold_fit_params.update(framework_params)

        fold_pipe.fit(X_tr, y_tr, **fold_fit_params)

        if calculate_importance:
            res = permutation_importance(
                fold_pipe,
                X_val,
                y_val,
                scoring="neg_root_mean_squared_error",
                n_repeats=perm_n_repeats,
                random_state=random_state,
                n_jobs=-1,
            )
            fold_importances.append(res.importances_mean)

        fold_preds_tr = fold_pipe.predict(X_tr)
        fold_preds_val = fold_pipe.predict(X_val)

        oof_preds_sum[val_idx] += fold_preds_val
        oof_counts[val_idx] += 1

        fitted_models.append(fold_pipe)

        tr_scores.append(root_mean_squared_error(y_tr, fold_preds_tr))
        val_scores.append(root_mean_squared_error(y_val, fold_preds_val))

    oof_preds = oof_preds_sum / oof_counts
    oof_mse = root_mean_squared_error(y_train, oof_preds)

    metrics = pd.DataFrame(
        [
            {
                "model": name,
                "TRAIN_rmsle_MEAN": np.mean(tr_scores),
                "TRAIN_rmsle_STD": np.std(tr_scores),
                "VAL_rmsle_MEAN": np.mean(val_scores),
                "VAL_rmsle_STD": np.std(val_scores),
                "OOF_rmsle": oof_mse,
            }
        ]
    )

    importance_df = None
    if calculate_importance:
        importance_df = pd.DataFrame(fold_importances, columns=X_train.columns).T
        fold_cols = importance_df.columns.tolist()

        importance_df["importance_mean"] = importance_df[fold_cols].mean(axis=1)
        importance_df["importance_std"] = importance_df[fold_cols].std(axis=1)

        importance_df = importance_df.sort_values(by="importance_mean", ascending=False)

        return fitted_models, oof_preds, metrics, val_scores, importance_df
    return fitted_models, oof_preds, metrics, val_scores

In [11]:
lgbm_search_model = lgb.LGBMRegressor(
    n_estimators=3000, learning_rate=0.05, verbosity=-1, random_state=gseed
)
es_callback = lgb.early_stopping(stopping_rounds=100, verbose=False)

### baseline with early stopping

In [116]:
models, oof, metrics, scores = cv_result(
    model=lgbm_search_model,
    X_train=X_train,
    y_train=y_train,
    y_strat=y_binned,
    cv_splitter=rskf,
    preprocessor=final_tree_pipe,
    name="baseline with early stopping",
    use_early_stopping=True,
    early_stopping_callback=es_callback,
)
metrics

baseline with early stopping:   0%|          | 0/50 [00:00<?, ?fold/s]

,model,TRAIN_rmsle_MEAN,TRAIN_rmsle_STD,VAL_rmsle_MEAN,VAL_rmsle_STD,OOF_rmsle
0,baseline with early stopping,0.037861,0.013322,0.119991,0.01287,0.117744


### linear regression

In [12]:
tree_pipe_for_lin = build_pipeline("final_tree_for_lin")

In [ ]:
models, oof, metrics, scores = cv_result(
    model=LinearRegression(),
    X_train=X_train,
    y_train=y_train,
    y_strat=y_binned,
    cv_splitter=rskf,
    preprocessor=tree_pipe_for_lin,
    name="linear regression",
    use_early_stopping=True,
)
metrics

linear regression:   0%|          | 0/50 [00:00<?, ?fold/s]

,model,TRAIN_rmsle_MEAN,TRAIN_rmsle_STD,VAL_rmsle_MEAN,VAL_rmsle_STD,OOF_rmsle
0,linear regression,0.096059,0.001321,0.114189,0.012617,0.113993


In [121]:
metrics_dict = metrics.iloc[0].to_dict()
log_fe_experiment(
    metrics=metrics_dict,
    note="tree pipe for lin + linear regression",
)

### ridge

In [123]:
models, oof, metrics, scores = cv_result(
    model=Ridge(),
    X_train=X_train,
    y_train=y_train,
    y_strat=y_binned,
    cv_splitter=rskf,
    preprocessor=tree_pipe_for_lin,
    name="ridge",
    use_early_stopping=True,
)
metrics

ridge:   0%|          | 0/50 [00:00<?, ?fold/s]

,model,TRAIN_rmsle_MEAN,TRAIN_rmsle_STD,VAL_rmsle_MEAN,VAL_rmsle_STD,OOF_rmsle
0,ridge,0.096086,0.001319,0.113783,0.012578,0.113622


In [124]:
metrics_dict = metrics.iloc[0].to_dict()
log_fe_experiment(
    metrics=metrics_dict,
    note="tree pipe for lin + ridge",
)

### lasso

In [ ]:
models, oof, metrics, scores = cv_result(
    model=Lasso(),
    X_train=X_train,
    y_train=y_train,
    y_strat=y_binned,
    cv_splitter=rskf,
    preprocessor=tree_pipe_for_lin,
    name="lasso",
    use_early_stopping=True,
)
metrics

ridge:   0%|          | 0/50 [00:00<?, ?fold/s]

,model,TRAIN_rmsle_MEAN,TRAIN_rmsle_STD,VAL_rmsle_MEAN,VAL_rmsle_STD,OOF_rmsle
0,ridge,0.399572,0.001523,0.399355,0.013719,0.399594


In [127]:
metrics_dict = metrics.iloc[0].to_dict()
log_fe_experiment(
    metrics=metrics_dict,
    note="tree pipe for lin + lasso",
)

### ElasticNet

In [129]:
models, oof, metrics, scores = cv_result(
    model=ElasticNet(),
    X_train=X_train,
    y_train=y_train,
    y_strat=y_binned,
    cv_splitter=rskf,
    preprocessor=tree_pipe_for_lin,
    name="elastic net",
    use_early_stopping=True,
)
metrics

elastic net:   0%|          | 0/50 [00:00<?, ?fold/s]

,model,TRAIN_rmsle_MEAN,TRAIN_rmsle_STD,VAL_rmsle_MEAN,VAL_rmsle_STD,OOF_rmsle
0,elastic net,0.399572,0.001523,0.399355,0.013719,0.399594


In [130]:
metrics_dict = metrics.iloc[0].to_dict()
log_fe_experiment(
    metrics=metrics_dict,
    note="tree pipe for lin + elastic net",
)

### knn

In [133]:
models, oof, metrics, scores = cv_result(
    model=KNeighborsRegressor(),
    X_train=X_train,
    y_train=y_train,
    y_strat=y_binned,
    cv_splitter=rskf,
    preprocessor=tree_pipe_for_lin,
    name="knn",
    use_early_stopping=True,
)
metrics

knn:   0%|          | 0/50 [00:00<?, ?fold/s]

,model,TRAIN_rmsle_MEAN,TRAIN_rmsle_STD,VAL_rmsle_MEAN,VAL_rmsle_STD,OOF_rmsle
0,knn,0.153807,0.001393,0.190454,0.015207,0.188216


In [134]:
metrics_dict = metrics.iloc[0].to_dict()
log_fe_experiment(
    metrics=metrics_dict,
    note="tree pipe for lin + knn",
)

### svr

In [136]:
models, oof, metrics, scores = cv_result(
    model=SVR(),
    X_train=X_train,
    y_train=y_train,
    y_strat=y_binned,
    cv_splitter=rskf,
    preprocessor=tree_pipe_for_lin,
    name="svr",
    use_early_stopping=True,
)
metrics

svr:   0%|          | 0/50 [00:00<?, ?fold/s]

,model,TRAIN_rmsle_MEAN,TRAIN_rmsle_STD,VAL_rmsle_MEAN,VAL_rmsle_STD,OOF_rmsle
0,svr,0.074769,0.000534,0.16439,0.018812,0.164294


In [137]:
metrics_dict = metrics.iloc[0].to_dict()
log_fe_experiment(
    metrics=metrics_dict,
    note="tree pipe for lin + svr",
)

### RandomForestRegressor

In [139]:
models, oof, metrics, scores = cv_result(
    model=RandomForestRegressor(),
    X_train=X_train,
    y_train=y_train,
    y_strat=y_binned,
    cv_splitter=rskf,
    preprocessor=tree_pipe_for_lin,
    name="random forest",
    use_early_stopping=True,
)
metrics

random forest:   0%|          | 0/50 [00:00<?, ?fold/s]

,model,TRAIN_rmsle_MEAN,TRAIN_rmsle_STD,VAL_rmsle_MEAN,VAL_rmsle_STD,OOF_rmsle
0,random forest,0.05146,0.000763,0.136159,0.013712,0.135158


In [140]:
metrics_dict = metrics.iloc[0].to_dict()
log_fe_experiment(
    metrics=metrics_dict,
    note="tree pipe for lin + random forest",
)

### Gradient Boosting Regressor

In [143]:
models, oof, metrics, scores = cv_result(
    model=GradientBoostingRegressor(
        n_estimators=1000,
        n_iter_no_change=100,
        validation_fraction=0.1,
        tol=1e-4,
        random_state=gseed,
    ),
    X_train=X_train,
    y_train=y_train,
    y_strat=y_binned,
    cv_splitter=rskf,
    preprocessor=tree_pipe_for_lin,
    name="sklearn gbdt",
    use_early_stopping=True,
)
metrics

sklearn gbdt:   0%|          | 0/50 [00:00<?, ?fold/s]

,model,TRAIN_rmsle_MEAN,TRAIN_rmsle_STD,VAL_rmsle_MEAN,VAL_rmsle_STD,OOF_rmsle
0,sklearn gbdt,0.062051,0.008198,0.122486,0.013372,0.118854


In [144]:
metrics_dict = metrics.iloc[0].to_dict()
log_fe_experiment(
    metrics=metrics_dict,
    note="tree pipe for lin + sklearn gbdt",
)

### hist gbdt

In [ ]:
models, oof, metrics, scores = cv_result(
    model=HistGradientBoostingRegressor(
        max_iter=1000, early_stopping=True, n_iter_no_change=100, random_state=gseed
    ),
    X_train=X_train,
    y_train=y_train,
    y_strat=y_binned,
    cv_splitter=rskf,
    preprocessor=tree_pipe_for_lin,
    name="sklearn hist gbdt",
    use_early_stopping=True,
)
metrics

sklearn hist gbdt:   0%|          | 0/50 [00:00<?, ?fold/s]

,model,TRAIN_rmsle_MEAN,TRAIN_rmsle_STD,VAL_rmsle_MEAN,VAL_rmsle_STD,OOF_rmsle
0,sklearn hist gbdt,0.020594,0.005361,0.123376,0.012607,0.120171


In [148]:
metrics_dict = metrics.iloc[0].to_dict()
log_fe_experiment(
    metrics=metrics_dict,
    note="tree pipe for lin + sklearn hist gbdt",
)

### CatBoost

In [153]:
models, oof, metrics, scores = cv_result(
    model=CatBoostRegressor(),
    X_train=X_train,
    y_train=y_train,
    y_strat=y_binned,
    cv_splitter=rskf,
    preprocessor=final_tree_pipe,
    name="catboost",
    use_early_stopping=True,
    fit_params={"model__cat_features": list(new_ohe_columns)},
)
metrics

catboost:   0%|          | 0/50 [00:00<?, ?fold/s]

,model,TRAIN_rmsle_MEAN,TRAIN_rmsle_STD,VAL_rmsle_MEAN,VAL_rmsle_STD,OOF_rmsle
0,catboost,0.056886,0.009948,0.113629,0.011599,0.11203


In [155]:
metrics_dict = metrics.iloc[0].to_dict()
log_fe_experiment(
    metrics=metrics_dict,
    note="final tree pipe + catboost",
)


### xgboost

In [172]:
models, oof, metrics, scores = cv_result(
    model=XGBRegressor(
        n_estimators=1000, early_stopping_rounds=100, random_state=gseed
    ),
    X_train=X_train,
    y_train=y_train,
    y_strat=y_binned,
    cv_splitter=rskf,
    preprocessor=final_tree_pipe,
    name="xgboost",
    use_early_stopping=True,
    fit_params={"model__verbose": 0},
)
metrics

xgboost:   0%|          | 0/50 [00:00<?, ?fold/s]

,model,TRAIN_rmsle_MEAN,TRAIN_rmsle_STD,VAL_rmsle_MEAN,VAL_rmsle_STD,OOF_rmsle
0,xgboost,0.020559,0.009931,0.132496,0.012468,0.123098


In [173]:
metrics_dict = metrics.iloc[0].to_dict()
log_fe_experiment(
    metrics=metrics_dict,
    note="final tree pipe + xgboost",
)

### check results

In [ ]:
log_file_path = Path(cfg.paths.logs) / "fe_experiments.log"
df = pd.read_json(log_file_path, lines=True)

normalized_data = pd.json_normalize(df["metrics"])
df = pd.concat([df.drop(columns=["metrics"]), normalized_data], axis=1)


with pd.option_context("display.max_colwidth", None, "display.max_columns", None):
    display(df.sort_values(by=["OOF_rmsle"], ascending=True))

,timestamp,note,model,TRAIN_rmsle_MEAN,TRAIN_rmsle_STD,VAL_rmsle_MEAN,VAL_rmsle_STD,OOF_rmsle
24,2026-09-18 22:52:28,final tree pipe + catboost,catboost,0.056886,0.009948,0.113629,0.011599,0.112030
16,2026-09-18 22:01:19,tree pipe for lin + ridge,ridge,0.096086,0.001319,0.113783,0.012578,0.113622
15,2026-09-18 21:58:06,tree pipe for lin + linear regression,linear regression,0.096059,0.001321,0.114189,0.012617,0.113993
13,2026-09-18 14:30:16,BWES + clean data + new lot frontage strategy + MonthFeatures + 22 features less,baseline with early stopping,0.037876,0.013941,0.119969,0.014396,0.117808
12,2026-09-18 00:51:50,"BWES + clean data + new lot frontage strategy + MonthFeatures, 5 features less",baseline with early stopping,0.040034,0.015401,0.120703,0.013265,0.118470
4,2026-09-17 18:35:15,BWES + clean data + new lot frontage strategy + MonthFeatures,baseline with early stopping,0.040034,0.015401,0.120703,0.013265,0.118470
7,2026-09-17 23:18:09,BWES + clean data + new lot frontage strategy + MonthFeatures + Flags,baseline with early stopping,0.040034,0.015401,0.120703,0.013265,0.118470
3,2026-09-17 18:16:37,BWES + clean data + new lot frontage strategy,baseline with early stopping,0.039861,0.014456,0.120734,0.013300,0.118564
5,2026-09-17 23:02:59,BWES + clean data + new lot frontage strategy + MonthFeatures + YearFeatures,baseline with early stopping,0.038167,0.013968,0.120806,0.012980,0.118629
14,2026-09-18 18:12:08,BWES + clean data + new lot frontage strategy + MonthFeatures + 25 features less,baseline with early stopping,0.038554,0.016160,0.120873,0.013886,0.118692


### tuning

* elastic net
* lgbm
* catboost

In [13]:
def run_one_fold(
    pipe,
    build_model_fn,
    params,
    X_tr,
    y_tr,
    X_val,
    y_val,
    fit_extra_fn=None,
    early_stopping=True,
):

    prep = clone(pipe)
    X_tr_t = prep.fit_transform(X_tr)
    X_val_t = prep.transform(X_val)

    model = build_model_fn(params)
    fit_kwargs = dict(fit_extra_fn(params) if fit_extra_fn else {})
    if early_stopping:
        fit_kwargs["eval_set"] = [(X_val_t, y_val)]

    model.fit(X_tr_t, y_tr, **fit_kwargs)

    rmse = root_mean_squared_error(y_val, model.predict(X_val_t))
    fitted_pipe = Pipeline([("preprocessor", prep), ("model", model)])
    return rmse, fitted_pipe

In [ ]:
def make_objective(
    sample_params_fn,
    build_model_fn,
    pipe,
    X,
    y,
    y_binned,
    cv,
    fit_extra_fn=None,
    early_stopping=True,
):
    def objective(trial):
        params = sample_params_fn(trial)
        fold_rmses = []
        fold_iter = enumerate(cv.split(X, y_binned))

        for step, (tr_idx, val_idx) in tqdm(
            fold_iter,
            total=cv.get_n_splits(),
            desc=f"trial {trial.number}",
            leave=False,
        ):
            X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
            y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

            rmse, _ = run_one_fold(
                pipe,
                build_model_fn,
                params,
                X_tr,
                y_tr,
                X_val,
                y_val,
                fit_extra_fn=fit_extra_fn,
                early_stopping=early_stopping,
            )
            fold_rmses.append(rmse)

            intermediate_value = float(np.mean(fold_rmses))
            trial.report(intermediate_value, step=step)

            if trial.should_prune():
                raise optuna.TrialPruned()

        trial.set_user_attr("fold_std", float(np.std(fold_rmses)))

        return float(np.mean(fold_rmses) + 0.5 * np.std(fold_rmses))

    return objective


In [15]:
def run_confirm_cv(
    pipe,
    build_model_fn,
    params,
    X,
    y,
    y_binned,
    cv,
    fit_extra_fn=None,
    early_stopping=True,
):
    fold_rmses, fitted_pipes = [], []
    for tr_idx, val_idx in tqdm(
        cv.split(X, y_binned), total=cv.get_n_splits(), desc="confirm cv"
    ):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        rmse, fitted_pipe = run_one_fold(
            pipe,
            build_model_fn,
            params,
            X_tr,
            y_tr,
            X_val,
            y_val,
            fit_extra_fn=fit_extra_fn,
            early_stopping=early_stopping,
        )
        fold_rmses.append(rmse)
        fitted_pipes.append(fitted_pipe)
    return fold_rmses, fitted_pipes

In [16]:
X_opt, X_hold, y_opt, y_hold, yb_opt, yb_hold = train_test_split(
    X_train,
    y_train,
    y_binned,
    test_size=0.15,
    stratify=y_binned,
    random_state=gseed,
)

yb_opt = pd.qcut(y_opt, q=10, labels=False)

rskf_search = RepeatedStratifiedKFold(n_splits=5, n_repeats=4, random_state=gseed)
rskf_confirm = RepeatedStratifiedKFold(n_splits=10, n_repeats=5, random_state=gseed)

In [17]:
def sample_params_en(trial):
    return {
        "alpha": trial.suggest_float("alpha", 1e-4, 10.0, log=True),
        "l1_ratio": trial.suggest_float("l1_ratio", 0.0, 1.0),
    }


def build_en(params):
    return ElasticNet(random_state=gseed, max_iter=20000, **params)


def sample_params_lgbm(trial):
    return {
        "num_leaves": trial.suggest_int("num_leaves", 7, 255, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "learning_rate": trial.suggest_float("learning_rate", 5e-3, 0.2, log=True),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "subsample_freq": trial.suggest_int("subsample_freq", 0, 7),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
    }


def build_lgbm(params):
    return lgb.LGBMRegressor(
        random_state=gseed, verbosity=-1, n_estimators=5000, **params
    )


def lgbm_fit_extra(params=None):
    return {"callbacks": [lgb.early_stopping(100, verbose=False)]}


def sample_params_cb(trial):
    return {
        "depth": trial.suggest_int("depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 1e-2, 0.3, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 30.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "border_count": trial.suggest_int("border_count", 32, 255),
    }


def build_cb(params):
    return CatBoostRegressor(
        random_state=gseed,
        loss_function="RMSE",
        iterations=5000,
        thread_count=1,
        **params,
    )


def cb_fit_extra(params=None):
    return {
        "cat_features": list(new_ohe_columns),
        "early_stopping_rounds": 100,
        "use_best_model": True,
        "verbose": False,
    }

In [18]:
output_dir = "optuna_results"
os.makedirs(output_dir, exist_ok=True)

storage = f"sqlite:///{output_dir}/optuna_studies.db"


sampler_en = optuna.samplers.TPESampler(seed=gseed, multivariate=True)
sampler_lgbm = optuna.samplers.TPESampler(seed=gseed, multivariate=True)
sampler_cb = optuna.samplers.TPESampler(seed=gseed, multivariate=True)

pruner_lgbm = optuna.pruners.MedianPruner(
    n_startup_trials=10, n_warmup_steps=3, interval_steps=1
)
pruner_cb = optuna.pruners.MedianPruner(
    n_startup_trials=10, n_warmup_steps=3, interval_steps=1
)

study_en = optuna.create_study(
    study_name="elastic_net",
    direction="minimize",
    sampler=sampler_en,
    storage=storage,
    load_if_exists=True,
)
study_lgbm = optuna.create_study(
    study_name="lightgbm",
    direction="minimize",
    sampler=sampler_lgbm,
    pruner=pruner_lgbm,
    storage=storage,
    load_if_exists=True,
)
study_cb = optuna.create_study(
    study_name="catboost",
    direction="minimize",
    sampler=sampler_cb,
    pruner=pruner_cb,
    storage=storage,
    load_if_exists=True,
)

[I 2026-09-19 16:17:35,519] Using an existing study with `study_name='elastic_net'` instead of creating a new one.
[I 2026-09-19 16:17:35,553] Using an existing study with `study_name='lightgbm'` instead of creating a new one.
[I 2026-09-19 16:17:35,584] Using an existing study with `study_name='catboost'` instead of creating a new one.


In [19]:
study_en.optimize(
    make_objective(
        sample_params_en,
        build_en,
        tree_pipe_for_lin,
        X_opt,
        y_opt,
        yb_opt,
        rskf_confirm,
        early_stopping=False,
    ),
    n_trials=100,
    n_jobs=8,
)

study_lgbm.optimize(
    make_objective(
        sample_params_lgbm,
        build_lgbm,
        final_tree_pipe,
        X_opt,
        y_opt,
        yb_opt,
        rskf_search,
        fit_extra_fn=lgbm_fit_extra,
    ),
    n_trials=50,
    n_jobs=8,
)

study_cb.optimize(
    make_objective(
        sample_params_cb,
        build_cb,
        final_tree_pipe,
        X_opt,
        y_opt,
        yb_opt,
        rskf_search,
        fit_extra_fn=cb_fit_extra,
    ),
    n_trials=25,
    n_jobs=8,
)

trial 6:   0%|          | 0/50 [00:00<?, ?it/s]

trial 9:   0%|          | 0/50 [00:00<?, ?it/s]

trial 8:   0%|          | 0/50 [00:00<?, ?it/s]

trial 7:   0%|          | 0/50 [00:00<?, ?it/s]

trial 10:   0%|          | 0/50 [00:00<?, ?it/s]

...

[I 2026-09-19 20:56:03,562] Trial 8 finished with value: 0.12607304109558487 and parameters: {'depth': 8, 'learning_rate': 0.025781645200306864, 'l2_leaf_reg': 11.126732970250512, 'random_strength': 0.16039820766801063, 'bagging_temperature': 0.24021576075549522, 'border_count': 88}. Best is trial 5 with value: 0.11818645486120531.


trial 23:   0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-09-19 21:02:21,528] Trial 21 finished with value: 0.12016046094044924 and parameters: {'depth': 3, 'learning_rate': 0.09332793616782045, 'l2_leaf_reg': 6.4169565047087795, 'random_strength': 1.307792441336554, 'bagging_temperature': 0.6373248893118116, 'border_count': 185}. Best is trial 5 with value: 0.11818645486120531.


trial 24:   0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-09-19 21:04:27,960] Trial 15 pruned. 
[I 2026-09-19 21:13:04,238] Trial 24 finished with value: 0.11832460836006281 and parameters: {'depth': 4, 'learning_rate': 0.050829777395433, 'l2_leaf_reg': 2.3465513693775724, 'random_strength': 3.2520729092817215, 'bagging_temperature': 0.425669607543883, 'border_count': 215}. Best is trial 5 with value: 0.11818645486120531.
[I 2026-09-19 21:13:49,852] Trial 20 finished with value: 0.1195833836304256 and parameters: {'depth': 5, 'learning_rate': 0.05888182388546897, 'l2_leaf_reg': 14.436638298759558, 'random_strength': 1.5155357530929872, 'bagging_temperature': 0.4443712001342771, 'border_count': 204}. Best is trial 5 with value: 0.11818645486120531.
[I 2026-09-19 21:19:47,478] Trial 22 finished with value: 0.11902296216560178 and parameters: {'depth': 4, 'learning_rate': 0.036851285951621134, 'l2_leaf_reg': 9.657900520924514, 'random_strength': 1.886610419452023, 'bagging_temperature': 0.5134813908797389, 'border_count': 237}. Best is t

In [20]:
en_best = study_en.best_params
lgbm_best = study_lgbm.best_params
cb_best = study_cb.best_params


In [21]:
rmses, fitted_pipes = run_confirm_cv(
    tree_pipe_for_lin,
    build_en,
    en_best,
    X_opt,
    y_opt,
    yb_opt,
    rskf_confirm,
    early_stopping=False,
)

hold_pred = np.mean([p.predict(X_hold) for p in fitted_pipes], axis=0)
hold_rmse = root_mean_squared_error(y_hold, hold_pred)
print(f"CV: {np.mean(rmses):.4f} +- {np.std(rmses):.4f}   Holdout: {hold_rmse:.4f}")

confirm cv:   0%|          | 0/50 [00:00<?, ?it/s]

CV: 0.1120 +- 0.0128   Holdout: 0.1150


In [22]:
rmses, fitted_pipes = run_confirm_cv(
    final_tree_pipe,
    build_lgbm,
    lgbm_best,
    X_opt,
    y_opt,
    yb_opt,
    rskf_confirm,
    fit_extra_fn=lgbm_fit_extra,
)

hold_pred = np.mean([p.predict(X_hold) for p in fitted_pipes], axis=0)
hold_rmse = root_mean_squared_error(y_hold, hold_pred)
print(f"CV: {np.mean(rmses):.4f} +- {np.std(rmses):.4f}   Holdout: {hold_rmse:.4f}")

confirm cv:   0%|          | 0/50 [00:00<?, ?it/s]

d:\vs_projects\fp_houses\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\vs_projects\fp_houses\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\vs_projects\fp_houses\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\vs_projects\fp_houses\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, ev

CV: 0.1136 +- 0.0145   Holdout: 0.1247


In [26]:
rmses, fitted_pipes = run_confirm_cv(
    final_tree_pipe,
    build_lgbm,
    {},
    X_opt,
    y_opt,
    yb_opt,
    rskf_confirm,
    fit_extra_fn=lgbm_fit_extra,
)

hold_pred = np.mean([p.predict(X_hold) for p in fitted_pipes], axis=0)
hold_rmse = root_mean_squared_error(y_hold, hold_pred)
print(f"CV: {np.mean(rmses):.4f} +- {np.std(rmses):.4f}   Holdout: {hold_rmse:.4f}")

confirm cv:   0%|          | 0/50 [00:00<?, ?it/s]

d:\vs_projects\fp_houses\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\vs_projects\fp_houses\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\vs_projects\fp_houses\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\vs_projects\fp_houses\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, ev

CV: 0.1231 +- 0.0158   Holdout: 0.1278


In [28]:
X_fit, X_es, y_fit, y_es = train_test_split(
    X_opt, y_opt, test_size=0.15, stratify=yb_opt, random_state=gseed
)
_, pipe = run_one_fold(
    final_tree_pipe,
    build_cb,
    cb_best,
    X_fit,
    y_fit,
    X_es,
    y_es,
    fit_extra_fn=cb_fit_extra,
)
hold_rmse = root_mean_squared_error(y_hold, pipe.predict(X_hold))
hold_rmse

0.12264591506367338

In [29]:
X_fit, X_es, y_fit, y_es = train_test_split(
    X_opt, y_opt, test_size=0.15, stratify=yb_opt, random_state=gseed
)
_, pipe = run_one_fold(
    final_tree_pipe,
    build_cb,
    {},
    X_fit,
    y_fit,
    X_es,
    y_es,
    fit_extra_fn=cb_fit_extra,
)
hold_rmse = root_mean_squared_error(y_hold, pipe.predict(X_hold))
hold_rmse

0.12510112913634927

### Итог
Подбор гиперпараметров дает прирост

Финальные классические модели:
* elasticnet - одна модель (все данные, нет early stopping, лучшие параметры)
* light gbm - собрать ансамбль из 50 моделей (все данные, лучшие параметры, early stopping в cv)
* catboost - одна модель (90% - под обучение, 10% - под остановку, лучшие параметры)